Connected to torch_env (Python 3.10.14)

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Siren modelled over the 2D Wave Equation as an Implicit Neural Rep. 

Equation: u_tt = D*(u_xx + u_yy), D=1.0

"""

# %%
configuration = {"Case": 'Wave',
                 "Field": 'u',
                 "Model": 'Siren',
                 "Epochs": 5,
                 "Batch Size": 200, #Actual batch will be Batch Size * Points
                 "Points": 1000, #Points sampled for from the grid.  
                 "Optimizer": 'Adam',
                 "Learning Rate": 0.005,
                 "Scheduler Step": 100,
                 "Scheduler Gamma": 0.5,
                 "Activation": 'GeLU',
                 "Physics Normalisation": 'No',
                 "Normalisation Strategy": 'Min-Max',
                 "T_range": 80, #Full range of time instances
                 "Layers": 5,                 
                 "Width": 256, 
                 "Coords": 3, #Number of spatio-temporal coordinates - would form the number of inputs 
                 "Variables":1, #Number of variables being modelled - would form the number of outputs. 
                 "Context": 100,
                 "Loss Function": 'MSE',
                 "UQ": 'None', #None, Dropout
                 }

In [2]:
import os
from simvue import Run
run = Run(mode='online')
run.init(folder="/Neural_PDE", tags=['NPDE', 'Siren', 'Tests', 'AR'], metadata=configuration)

# Saving the current run file and the git hash of the repo
run.save(os.path.abspath(__file__), 'code')
import git
repo = git.Repo(search_parent_directories=True)
sha = repo.head.object.hexsha
run.update_metadata({'Git Hash': sha})

[simvue] Run sour-canoe created
[simvue] Monitor in the UI at https://dev02.simvue.io/dashboard/runs/run/Qvb4LGk5EHJmQSxRcCNQJT


True

In [3]:
#Importing the necessary packages
import sys
import numpy as np
from tqdm import tqdm 
import torch
import matplotlib
import matplotlib.pyplot as plt
import time 
from timeit import default_timer
from tqdm import tqdm 

#Adding the NPDE package to the system python path
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

In [4]:
#Importing the models and utilities. 
from Neural_PDE.Models.INR import *
from Neural_PDE.Utils.processing_utils import * 
from Neural_PDE.Utils.training_utils import * 

In [5]:
#Settung up locations. 
file_loc = os.getcwd()
data_loc = os.path.dirname(os.getcwd()) + '/Data'
model_loc = file_loc + '/Weights'
plot_loc = file_loc + '/Plots'
#Setting up the seeds and devices
torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
################################################################
# Loading Data 
################################################################

In [7]:
t1 = default_timer()
data =  np.load(data_loc + '/Spectral_Wave_data_LHS.npz')

u_sol = data['u'].astype(np.float32)
x = data['x'].astype(np.float32)
y = data['y'].astype(np.float32)
t = data['t'].astype(np.float32)
u = torch.from_numpy(u_sol)
u = u.permute(0, 2, 3, 1)
n_sims = len(u_sol)

In [8]:
ntrain = 800
ntest = 200
S = 33 #Grid Size

#Extracting configuration files
num_coords = configuration['Coords']
num_vars = configuration['Variables']
layers = configuration['Layers']
width = configuration['Width']
batch_size = configuration['Batch Size']
T_range = configuration['T_range']
num_points = configuration["Points"]

#Slicing the fields and setting up the coordinate meshes. 
t = t[:T_range]
u = u[...,:T_range]
xx, yy, tt = np.meshgrid(x, y, t)

#Stacked cooredinate values and corresponding field values stacked. 
aa = np.vstack((xx.flatten(), yy.flatten(), tt.flatten() )).T
uu = u.reshape(u.shape[0], int(u.shape[1]*u.shape[2]*u.shape[3])).unsqueeze(-1)

#coordinates 
coords = torch.tensor(aa, dtype=torch.float32)

#Getting the context from the initial conditions
context_len = configuration['Context']
aa_context = sample_equidistant(u[...,0], context_len)

NameError: name 'sample_equidistant' is not defined

In [9]:
ntrain = 800
ntest = 200
S = 33 #Grid Size

#Extracting configuration files
num_coords = configuration['Coords']
num_vars = configuration['Variables']
layers = configuration['Layers']
width = configuration['Width']
batch_size = configuration['Batch Size']
T_range = configuration['T_range']
num_points = configuration["Points"]

#Slicing the fields and setting up the coordinate meshes. 
t = t[:T_range]
u = u[...,:T_range]
xx, yy, tt = np.meshgrid(x, y, t)

#Stacked cooredinate values and corresponding field values stacked. 
aa = np.vstack((xx.flatten(), yy.flatten(), tt.flatten() )).T
uu = u.reshape(u.shape[0], int(u.shape[1]*u.shape[2]*u.shape[3])).unsqueeze(-1)

#coordinates 
coords = torch.tensor(aa, dtype=torch.float32)

#Getting the context from the initial conditions

def sample_equidistant(grid, num_samples):
   
    #    Sample values from a 2D NumPy grid in an equidistant manner.

    # Args:
    #     grid (numpy.ndarray): A 3D NumPy array representing the num_sim, x_grid, y_grid
    #     num_samples (int): The total number of samples to retrieve equidistantly along each axis.

    # Returns:
    #     numpy.ndarray: A 3D NumPy array containing the sampled values from the equidistant points

    sims, height, width = grid.shape
    num_samples = int(np.sqrt(num_samples))
    # Generate equidistant x-coordinates
    x_coords = np.linspace(0, width - 1, num_samples, dtype=int)
    
    # Generate equidistant y-coordinates
    y_coords = np.linspace(0, height - 1, num_samples, dtype=int)
    
    # Create a meshgrid of the x and y coordinates
    xx, yy = np.meshgrid(x_coords, y_coords)
    
    # Flatten the meshgrid to get the indices
    indices = np.vstack((yy.flatten(), xx.flatten())).T
    
    # Sample the grid using the indices
    samples = grid[:, indices[:, 0], indices[:, 1]]
    
    return samples
context_len = configuration['Context']
aa_context = sample_equidistant(u[...,0], context_len)

In [10]:
aa_context.shape

torch.Size([1000, 100])